# Car Price Prediction with Machine Learning

This notebook builds a used car price prediction model using the CarDekho dataset.

In [ ]:
import os
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

plt.style.use('seaborn-v0_8')
sns.set_theme(style='whitegrid')

In [ ]:
# Load the dataset
dataset_path = 'car_data.csv'

if not os.path.exists(dataset_path):
    raise FileNotFoundError('Please download the CarDekho dataset and save it as car_data.csv in this folder.')

df = pd.read_csv(dataset_path)

print(df.head())
print('\nDataset shape:', df.shape)
print('\nInfo:')
df.info()
print('\nSummary statistics:')
df.describe(include='all')

In [ ]:
# Data cleaning
df.isnull().sum()

df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

# Standardize categorical values
df['Fuel_Type'] = df['Fuel_Type'].astype(str).str.lower().str.strip()
df['Seller_Type'] = df['Seller_Type'].astype(str).str.lower().str.strip()
df['Transmission'] = df['Transmission'].astype(str).str.lower().str.strip()

print('Cleaned dataset shape:', df.shape)

In [ ]:
# Feature engineering
current_year = 2025
df['Car_Age'] = current_year - df['Year']
df['Brand'] = df['Car_Name'].astype(str).apply(lambda x: x.split()[0])

df[['Car_Name', 'Year', 'Car_Age', 'Brand']].head()

In [ ]:
# EDA - Distribution of selling price
plt.figure(figsize=(8, 5))
sns.histplot(df['Selling_Price'], bins=30, kde=True)
plt.title('Selling Price Distribution')
plt.show()

In [ ]:
# EDA - Price vs Fuel Type
plt.figure(figsize=(8, 5))
sns.boxplot(x='Fuel_Type', y='Selling_Price', data=df)
plt.title('Selling Price by Fuel Type')
plt.show()

In [ ]:
# EDA - Price vs Car Age
plt.figure(figsize=(8, 5))
sns.scatterplot(x='Car_Age', y='Selling_Price', data=df)
plt.title('Selling Price vs Car Age')
plt.show()

In [ ]:
# Prepare features and target
X = df.drop('Selling_Price', axis=1)
y = df['Selling_Price']

X = X.drop('Car_Name', axis=1)

categorical = ['Fuel_Type', 'Seller_Type', 'Transmission', 'Brand']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)
    ],
    remainder='passthrough'
)

X.head()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Train models
lr_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

rf_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

gb_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(random_state=42))
])

lr_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)
gb_model.fit(X_train, y_train)

In [ ]:
# Evaluate models
def evaluate_model(model, name):
    pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    print(f'{name}:')
    print('MAE :', mae)
    print('RMSE:', rmse)
    print('R2  :', r2)
    print('-' * 40)
    return mae, rmse, r2

lr_mae, lr_rmse, lr_r2 = evaluate_model(lr_model, 'Linear Regression')
rf_mae, rf_rmse, rf_r2 = evaluate_model(rf_model, 'Random Forest')
gb_mae, gb_rmse, gb_r2 = evaluate_model(gb_model, 'Gradient Boosting')

In [ ]:
# Compare model performance
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'Gradient Boosting'],
    'MAE': [lr_mae, rf_mae, gb_mae],
    'RMSE': [lr_rmse, rf_rmse, gb_rmse],
    'R2': [lr_r2, rf_r2, gb_r2]
})

results.sort_values(by='R2', ascending=False)

In [ ]:
# Feature importance (Random Forest)
encoded_features = rf_model.named_steps['preprocessor'].get_feature_names_out()
importances = rf_model.named_steps['model'].feature_importances_

importance_df = pd.DataFrame({
    'Feature': encoded_features,
    'Importance': importances
})

importance_df = importance_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df.head(15))
plt.title('Top 15 Important Features')
plt.show()

## Conclusion

In this project, a used car price prediction system was developed using machine learning. The dataset was cleaned by handling missing values, removing duplicates, and standardizing categorical values. Feature engineering included calculating car age and extracting the vehicle brand. Exploratory Data Analysis (EDA) helped visualize price distributions and relationships with fuel type and car age. Three regression models—Linear Regression, Random Forest Regressor, and Gradient Boosting Regressor—were trained and evaluated using MAE, RMSE, and R² score. Among these, the model with the highest R² score and lowest error metrics can be selected as the best-performing model. Feature importance analysis identifies the variables that most influence used car prices.